<a href="https://colab.research.google.com/github/olhacherenkova/User-Analysis-of-Calorie-Tracker-/blob/main/TeamProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [20]:
# Створення дата сету з рандомних даних для прорахунку формул
n = 956
# Стать
gender = np.random.choice(["Male", "Female"], size=n, p=[0.5, 0.5])
# Вік
age =  np.random.randint(18, 70, size=n)
# Зріст
height = np.where(gender == "Male",
                  np.random.normal(175 , 7, n),
                  np.random.normal(162, 6, n)
                  )
# Індекс маси тіла
bmi = np.random.normal(23, 3, n)
# Вага
weight = bmi * (height/100)**2
# Відсоток жиру
sex_numeric = np.where(gender=="Male", 1, 0)
bfat = (1.20 * bmi) + (0.23 * age) - (10.8 * sex_numeric) - 5.4
# Уподобання у їжі
food_preferences = np.random.choice(["Vegan", "Vegetarian","Keto", "Low‑Carb / LCHF", "High‑Protein", "Gluten‑Free", "Lactose‑Free", "Omnivore", "Sugar‑Free Diet"],size=n)
# Середня кількість відкриття додатку в день
app_open_count_day = np.random.randint(0, 10, size=n)
# Середня кількість внесених страв в день
log_meals_count = np.where(app_open_count_day == 0,
                  0,
                  np.random.randint(0, 5, size=n)
                  )
# Середня кількість активних днів
days_active = np.random.randint(1, 120, size=n)
# Середня тривалість сесії
session_duration = np.where(app_open_count_day == 0,
                  0,
                  np.random.randint(5, 300, size=n)
                  )
# Операціна система
os_platform =  np.random.choice(["iOS","Android"],size=n, p=[0.67, 0.33])
# Країна
country = np.random.choice( [
    "United States", "Canada", "Mexico", "Brazil", "Argentina",
    "United Kingdom", "Germany", "France", "Italy", "Spain",
    "Ukraine", "Poland", "Czech Republic", "Netherlands", "Sweden",
    "Norway", "Denmark", "Finland", "Turkey",
    "China", "Japan", "South Korea", "India", "Indonesia",
    "Australia", "New Zealand", "South Africa", "Egypt", "Nigeria"
], size=n)
# Тип підписки
subscription_type = np.random.choice(["Premium","Free"],size=n, p=[0.28, 0.72])

df = pd.DataFrame({
    "id": range(1, n+1),
    "age": age,
    "gender": gender,
    "height": height.round(),
    "weight": weight.round(1),
    "bfat":bfat.round(1),
    "bmi": bmi.round(),
    "food_preferences": food_preferences,
    "avg_app_open_count_day": app_open_count_day,
    "avg_log_meals_count": log_meals_count,
    "days_active" : days_active,
    "avg_session_duration" : session_duration,
    "os_platform" : os_platform,
    "country": country,
"subscription_type":subscription_type

})

# Функція для визначення activity_lvl
def assign_activity(bmi):
    if bmi < 20:
        return np.random.choice(["exhausting", "moderate"], p=[0.7, 0.3])
    elif bmi < 25:
        return np.random.choice(["moderate", "high"], p=[0.6, 0.4])
    elif bmi < 30:
        return np.random.choice(["moderate", "static"], p=[0.5, 0.5])
    else:
        return "static"

df["activity_lvl"] = df["bmi"].apply(assign_activity)

# Функція для визначення categories
def assign_category(activity):
    if activity == "static":
        return np.random.choice(["weight loss", "general"], p=[0.7, 0.3])
    elif activity == "moderate":
        return np.random.choice(["general", "weight loss"], p=[0.5, 0.5])
    elif activity == "high":
        return np.random.choice(["general", "recomposition"], p=[0.6, 0.4])
    else:  # exhausting
        return np.random.choice(["weight gain", "recomposition"], p=[0.6, 0.4])

df["categories"] = df["activity_lvl"].apply(assign_category)


print(df)


      id  age  gender  height  weight  bfat   bmi food_preferences  \
0      1   29    Male   180.0    77.2  19.1  24.0             Keto   
1      2   66    Male   178.0    90.7  33.2  29.0      Gluten‑Free   
2      3   57    Male   178.0    74.0  24.9  23.0         Omnivore   
3      4   19  Female   158.0    59.9  27.7  24.0     Lactose‑Free   
4      5   55    Male   177.0    86.8  29.8  28.0  Sugar‑Free Diet   
..   ...  ...     ...     ...     ...   ...   ...              ...   
951  952   46  Female   144.0    38.7  27.4  19.0             Keto   
952  953   58  Female   158.0    46.4  30.2  19.0            Vegan   
953  954   49    Male   169.0    65.4  22.6  23.0         Omnivore   
954  955   24  Female   162.0    50.5  23.2  19.0     Lactose‑Free   
955  956   55  Female   169.0    80.3  41.0  28.0     High‑Protein   

     avg_app_open_count_day  avg_log_meals_count  days_active  \
0                         9                    4           72   
1                         0  

In [21]:
# Збереження датафрейму у CSV
df.to_csv("results.csv", index=False, encoding="utf-8")
from google.colab import files
files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Формула BMR (Basal Metabolic Rate)
def calculate_bmr(row):
    if row['gender'] == 'M' and row['categories'] == 'general' :  # чоловіки
        return 10 * row['weight'] + 6.25 * row['height'] - 5 * row['age'] + 5
    elif row['gender'] == 'F' and row['categories'] == 'general' :  # жінки
        return 10 * row['weight'] + 6.25 * row['height'] - 5 * row['age'] - 161
    else:
        return 370 + (21.6 * (row['weight'] * ( 1 - row['bfat'] / 100)))

# Коефіцієнти активності
activity_map = {
    "static": 1.2,
    "mild": 1.375,
    "moderate":1.55,
    "high":1.725,
    "exhausting":1.9
}

# Додаємо колонку з BMR
df['BMR'] = df.apply(calculate_bmr, axis=1)

# Додаємо колонку з TDEE (Total Daily Energy Expenditure)
df['calories'] = df.apply(lambda row: round(row['BMR'] * activity_map.get(row['activity_lvl'], 1.2),0), axis=1)

def calculate_protein(row):
    if row['categories'] == 'general' :
        return round(1 * row['weight'])
    elif row['categories'] == 'weight loss' :
        return round(2 * row['weight']*(1 - row['bfat']/100))
    elif row['categories'] == 'weight gain' :
        return round(1.7 * row['weight']*(1 - row['bfat']/100))
    else:
        return round(1.9 * row['weight']*(1 - row['bfat']/100))

df['protein'] = df.apply(calculate_protein, axis=1)

def calculate_fats(row):
    if row['categories'] == 'general' :
        return round(0.3 * row['calories'] / 9)
    elif row['categories'] == 'recomposition' :
        return round(0.28 * row['calories'] / 9)
    else:
        return round(0.25 * row['calories'] / 9)

df['fats'] = df.apply(calculate_fats, axis=1)

df['carbs'] = df.apply(lambda row: round((row['calories'] - row['protein'] * 4 - row['fats'] * 9)/4), axis=1)

df['calories_right'] = df.apply(lambda row: row['protein'] * 4 + row['fats'] * 9 + row['carbs'] * 4, axis=1)

print(df)



      id  age  gender  height  weight  bfat   bmi food_preferences  \
0      1   56    Male   173.0    82.2  29.7  28.0             Keto   
1      2   44    Male   178.0    49.3  12.6  16.0     High‑Protein   
2      3   32    Male   173.0    69.5  19.1  23.0            Vegan   
3      4   38    Male   171.0    61.0  17.5  21.0     High‑Protein   
4      5   47    Male   185.0    66.6  18.0  19.0  Sugar‑Free Diet   
..   ...  ...     ...     ...     ...   ...   ...              ...   
951  952   62    Male   176.0    63.7  22.6  20.0  Low‑Carb / LCHF   
952  953   18  Female   161.0    60.6  26.8  23.0            Vegan   
953  954   55  Female   153.0    53.7  34.8  23.0  Sugar‑Free Diet   
954  955   58  Female   158.0    50.4  32.1  20.0     High‑Protein   
955  956   50  Female   174.0    68.0  33.2  23.0       Vegetarian   

    activity_lvl   categories         BMR  calories  protein  fats  carbs  \
0         static  weight loss  1618.19056    1942.0      116    54    248   
1    